# Geometry Of Data Homework 3
## Manifold Learning with Autoencoders
### By: Sravanth Chowdary Potluri und5uv

### Part 1
Download the code Autoencoder.ipynb. You will want to copy the code for the model definitions, but do not run the training! (unless you have a GPU and want to play with the models beyond the assignment) Download the pre-trained fully-connected layer models fcAE<dim>.pth and convolutional layer models convAE<dim>.pth, where <dim> is the code (z) dimension: 16, 32, or 128.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import FashionMNIST
import torchvision.utils as vutils
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# Set up device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

# Load test data
batch_size = 100
test_data = FashionMNIST("./data", train=False, download=True,
                         transform=transforms.ToTensor())
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
# Function to plot images
def plot_images(batch, rows, cols, title=""):
    plt.figure(figsize=(cols, rows))
    plt.axis("off")
    plt.title(title)
    plt.imshow(np.transpose(
        vutils.make_grid(batch[:(rows * cols)], nrow=rows, normalize=True).cpu(),
        (1, 2, 0)))
    plt.show()

In [ ]:
# Define model classes
class FullyConnectedAutoencoder(nn.Module):
    def __init__(self, out_size):
        super(FullyConnectedAutoencoder, self).__init__()
        self.elayer1 = nn.Linear(28*28, 256)
        self.ebatch1 = nn.BatchNorm1d(256)
        self.elayer2 = nn.Linear(256, 128)
        self.ebatch2 = nn.BatchNorm1d(128)
        self.elayer3 = nn.Linear(128, out_size)

        self.dlayer1 = nn.Linear(out_size, 128)
        self.dbatch1 = nn.BatchNorm1d(128)
        self.dlayer2 = nn.Linear(128, 256)
        self.dbatch2 = nn.BatchNorm1d(256)
        self.dlayer3 = nn.Linear(256, 28*28)

    def encoder(self, x):
        y1 = F.elu(self.ebatch1(self.elayer1(x)))
        y2 = F.elu(self.ebatch2(self.elayer2(y1)))
        z = F.elu(self.elayer3(y2))
        return z

    def decoder(self, z):
        y1 = F.elu(self.dbatch1(self.dlayer1(z)))
        y2 = F.elu(self.dbatch2(self.dlayer2(y1)))
        x = torch.sigmoid(self.dlayer3(y2))
        return x

    def forward(self, x):
        x = x.view(-1, 28*28)
        z = self.encoder(x)
        y = self.decoder(z)
        y = y.view(-1, 1, 28, 28)
        return y

class ConvolutionalAutoencoder(nn.Module):
    def __init__(self, out_size):
        super(ConvolutionalAutoencoder, self).__init__()
        self.econv1 = nn.Conv2d(1, 32, kernel_size=5, bias=False)
        self.ebatch1 = nn.BatchNorm2d(32)
        self.econv2 = nn.Conv2d(32, 8, kernel_size=5, bias=False)
        self.ebatch2 = nn.BatchNorm2d(8)
        self.econv3 = nn.Conv2d(8, out_size, kernel_size=20, bias=True)

        self.dconv1 = nn.ConvTranspose2d(out_size, 8, kernel_size=20, bias=True)
        self.dbatch1 = nn.BatchNorm2d(8)
        self.dconv2 = nn.ConvTranspose2d(8, 32, kernel_size=5, bias=False)
        self.dbatch2 = nn.BatchNorm2d(32)
        self.dconv3 = nn.ConvTranspose2d(32, 1, kernel_size=5, bias=False)

    def encoder(self, x):
        y1 = F.elu(self.ebatch1(self.econv1(x)))
        y2 = F.elu(self.ebatch2(self.econv2(y1)))
        z = F.elu(self.econv3(y2))
        return z

    def decoder(self, z):
        y1 = F.elu(self.dbatch1(self.dconv1(z)))
        y2 = F.elu(self.dbatch2(self.dconv2(y1)))
        x = torch.sigmoid(self.dconv3(y2))
        return x

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y

In [ ]:
# Load models
models = {}

# Fully Connected Autoencoders
fcAE16 = FullyConnectedAutoencoder(16)
fcAE16.load_state_dict(torch.load("hw3/fcAE16.pth", map_location=device))
fcAE16 = fcAE16.to(device)
fcAE16.eval()
models['fcAE16'] = fcAE16

fcAE32 = FullyConnectedAutoencoder(32)
fcAE32.load_state_dict(torch.load("hw3/fcAE32.pth", map_location=device))
fcAE32 = fcAE32.to(device)
fcAE32.eval()
models['fcAE32'] = fcAE32

fcAE128 = FullyConnectedAutoencoder(128)
fcAE128.load_state_dict(torch.load("hw3/fcAE128.pth", map_location=device))
fcAE128 = fcAE128.to(device)
fcAE128.eval()
models['fcAE128'] = fcAE128

# Convolutional Autoencoders
convAE16 = ConvolutionalAutoencoder(16)
convAE16.load_state_dict(torch.load("hw3/convAE16.pth", map_location=device))
convAE16 = convAE16.to(device)
convAE16.eval()
models['convAE16'] = convAE16

convAE32 = ConvolutionalAutoencoder(32)
convAE32.load_state_dict(torch.load("hw3/convAE32.pth", map_location=device))
convAE32 = convAE32.to(device)
convAE32.eval()
models['convAE32'] = convAE32

convAE128 = ConvolutionalAutoencoder(128)
convAE128.load_state_dict(torch.load("hw3/convAE128.pth", map_location=device))
convAE128 = convAE128.to(device)
convAE128.eval()
models['convAE128'] = convAE128

### Part 2
Plot a 10 ×10 grid of images from the test set. For each model, plot the same grid of
these images reconstructed by that model. (Recall reconstructing an image means running
it through the encoder followed by the decoder). Qualitatively compare the various models.
What difference does convolutional vs. fully-connected layers make? What difference does
the code dimension make?

In [ ]:
# Step 2: Plot original and reconstructed images
first_batch = next(iter(test_loader))
images, labels = first_batch
images = images.to(device)

# Plot original images
plot_images(images.cpu(), 10, 10, "Original Test Images")

# Reconstruct and plot images for each model
for model_name, model in models.items():
    with torch.no_grad():
        reconstructed_images = model(images)
    plot_images(reconstructed_images.cpu(), 10, 10, f"Reconstructed Images - {model_name}")


#### The difference between the models with fully connected layers and the convolution layers is clearly visible to the naked eye. The fully connected layers are not able to capture the finer details in the images resulting in a reconstruction that is not as detailed as the original image. This image contains only the basic features of the original image such as shape. However the models with convolutional layers show the reconstructed images with much more detail. The convolutional layers are able to capture the finer details in the images and the reconstructed images are much closer to the original images. The difference in the code dimension is also visible in the reconstructed images. The models with a higher code dimension are able to capture more details in the images and the reconstructed images are closer to the original images.

#### The difference between the dimensions of the model is more obvious in the case of the convolutional autoencoders as compared to the fully connected autoencoders. The convolutional autoencoders with a higher code dimension are able to capture more details in the images and the reconstructed images are closer to the original images. You can really see this in the second image of the t-shirt with some text on it. The convolutional model with the highest dimension begins to clearly show the text while the smaller models do not. Unfortunately there are only minute differences in the images between the different sized fully connected autoencoders. However you can see some very fine detail in the highest dimension model that is not present in the lower dimension models, i.e the model becomes more sharp as the dimension increases.

### Part 3
Compute a PCA of the training images. Plot the same 10×10 images reconstructed from the
first k principal component dimensions, where k = 16, 32, 128. How does PCA reconstruction
compare qualitatively to the autoencoders? What does this tell you about the data manifold?

In [ ]:
# Step 3: PCA reconstruction
# Load training data
train_data = FashionMNIST("./data", train=True, download=True,
                          transform=transforms.ToTensor())
train_images = train_data.data  # [N, H, W]
N, H, W = train_images.shape
D = H * W
train_images_flat = train_images.view(N, -1).float() / 255.0  # [N, D]

# Prepare test images
test_images = test_data.data[:100]  # [100, H, W]
test_images_flat = test_images.view(100, -1).float() / 255.0  # [100, D]

# Fit PCA on training data
pca = PCA()
pca.fit(train_images_flat.numpy())

# Reconstruct test images using PCA
for k in [16, 32, 128]:
    pca_k = PCA(n_components=k)
    pca_k.fit(train_images_flat.numpy())
    test_images_pca = pca_k.transform(test_images_flat.numpy())
    test_images_reconstructed = pca_k.inverse_transform(test_images_pca)
    test_images_reconstructed = test_images_reconstructed.reshape(100, H, W)
    # Convert to tensor
    test_images_reconstructed = torch.tensor(test_images_reconstructed).unsqueeze(1)
    plot_images(test_images_reconstructed, 10, 10, f"PCA Reconstruction with k={k}")

#### The PCA reconstruction is not as good as the autoencoders reconstruction at especially smaller k-values, However it starts to exhibit much more detail at the larger k=128.One of the key details this gives us about the data manifold is that the data lies on a manifold that is on a larger dimension than 128 dimensions.Even though the PCA and autoencoders are able to capture the finer nuances of the data at 128 dimensions, it can still be better. Another observation is that the autoencoders are able to capture the finer details in the images and the reconstructed images are much closer to the original images. The PCA reconstruction is not as good as the autoencoders reconstruction at especially smaller k-values. This could be because of the non-linear nature of the data which PCA cannot capture in lesser dimensions. The autoencoders are able to capture the non-linear nature of the data better and hence are able to reconstruct the images better. 

### Part 4
Pick two images from the test set that visually seem very different (e.g., from two different
classes). Plot a sequence of images that are along the linear path between these two images.
Next, for each autoencoder model, plot a sequence of images that are interpolated in the
latent code space, followed by application of the decoder. Try all these plots again with
two images that look visually similar. Describe what you see comparing all of the different
interpolations.

In [ ]:
# Step 4: Interpolation between two images
# Pick two different images
test_labels = test_data.targets
indices_class0 = (test_labels == 0).nonzero(as_tuple=False).squeeze()
indices_class1 = (test_labels == 1).nonzero(as_tuple=False).squeeze()
i1 = indices_class0[0].item()
i2 = indices_class1[0].item()

img1, _ = test_data[i1]
img2, _ = test_data[i2]
img1 = img1.unsqueeze(0).to(device)
img2 = img2.unsqueeze(0).to(device)

n_steps = 10
alphas = np.linspace(0, 1, n_steps)

# Interpolation in pixel space
interpolated_images = []
for alpha in alphas:
    interpolated_image = (1 - alpha) * img1 + alpha * img2
    interpolated_images.append(interpolated_image)
interpolated_images = torch.cat(interpolated_images, dim=0)
plot_images(interpolated_images.cpu(), n_steps, 10, "Linear Interpolation in Pixel Space")

# Interpolation in latent space for each model
for model_name, model in models.items():
    model.eval()
    with torch.no_grad():
        if isinstance(model, FullyConnectedAutoencoder):
            img1_input = img1.view(-1, 28*28)
            img2_input = img2.view(-1, 28*28)
            z1 = model.encoder(img1_input)
            z2 = model.encoder(img2_input)
        else:
            z1 = model.encoder(img1)
            z2 = model.encoder(img2)
    interpolated_z = []
    for alpha in alphas:
        z = (1 - alpha) * z1 + alpha * z2
        interpolated_z.append(z)
    if isinstance(model, FullyConnectedAutoencoder):
        interpolated_z = torch.cat(interpolated_z, dim=0)
        with torch.no_grad():
            y = model.decoder(interpolated_z)
            y = y.view(-1, 1, 28, 28)
    else:
        interpolated_z = torch.cat(interpolated_z, dim=0)
        with torch.no_grad():
            y = model.decoder(interpolated_z)
    plot_images(y.cpu(), n_steps, 10, f"Latent Space Interpolation - {model_name}")

#### The linear interpolation in pixel space shows a simple linear interpolation between the two images, with each pixel fading of one image fading and the images from the other pixel appearing. However when you plot the sequence of images that are interpolated using the latent variables in the latent space and then reconstructed, you can see a much more gradual change with the image slowly deforming from the initial image to the final image. Interestingly the deformation seems more cleaner in the model with the lesser latent dimension, this could be because of the lower dimensions approximating the images more closely.

In [ ]:
# now picking two images from the same class and doing the same interpolation
indices_class0 = (test_labels == 2).nonzero(as_tuple=False).squeeze()
indices_class1 = (test_labels == 2).nonzero(as_tuple=False).squeeze()
i1 = indices_class0[0].item()
i2 = indices_class0[1].item()

img1, _ = test_data[i1]
img2, _ = test_data[i2]
img1 = img1.unsqueeze(0).to(device)
img2 = img2.unsqueeze(0).to(device)

n_steps = 10
alphas = np.linspace(0, 1, n_steps)

# Interpolation in pixel space
interpolated_images = []
for alpha in alphas:
    interpolated_image = (1 - alpha) * img1 + alpha * img2
    interpolated_images.append(interpolated_image)
interpolated_images = torch.cat(interpolated_images, dim=0)
plot_images(interpolated_images.cpu(), n_steps, 10, "Linear Interpolation in Pixel Space")

# Interpolation in latent space for each model
for model_name, model in models.items():
    model.eval()
    with torch.no_grad():
        if isinstance(model, FullyConnectedAutoencoder):
            img1_input = img1.view(-1, 28*28)
            img2_input = img2.view(-1, 28*28)
            z1 = model.encoder(img1_input)
            z2 = model.encoder(img2_input)
        else:
            z1 = model.encoder(img1)
            z2 = model.encoder(img2)
    interpolated_z = []
    for alpha in alphas:
        z = (1 - alpha) * z1 + alpha * z2
        interpolated_z.append(z)
    if isinstance(model, FullyConnectedAutoencoder):
        interpolated_z = torch.cat(interpolated_z, dim=0)
        with torch.no_grad():
            y = model.decoder(interpolated_z)
            y = y.view(-1, 1, 28, 28)
    else:
        interpolated_z = torch.cat(interpolated_z, dim=0)
        with torch.no_grad():
            y = model.decoder(interpolated_z)
    plot_images(y.cpu(), n_steps, 10, f"Latent Space Interpolation - {model_name}")

#### in this case where we try to interpolate between two images of the same class, the sequence of images coming from the linear interpolations seem similar to the ones generated by the latent space representation. However this could be because the shape is not changing much between the two images. Here, similar to the before scenarios, the models with convolution layers and more layers seem to have capturing the data to a more lower level causing the transformation to be more finely detailed.

### Part 5
For each model, answer the following questions:

(a) What are the domains and ranges for each layer? (They are all mappings from $\mathbb{R}^a$ to $\mathbb{R}^b$,
so this is asking what is the input dimension a and output dimension b for each layer.)

(b) By looking only at the weights, is the encoder a submersion? Possible answers are “yes”,
“no”, “can’t tell from the weights alone”. Explain why.

(c) Again, looking only at the weights, is the decoder an immersion? Same possible answers
as above. Explain why.


In [ ]:
# Display all the layers and their dimensions for each model, for each layer in each model, We just print the model information
for model_name, model in models.items():
    print(f"Model: {model_name}")
    for name, param in model.named_parameters():
        print(f"Layer: {name}")
        print(f"Shape: {param.shape}")
        print()
    print()

#### The Input and output dimensions for each layer in each model are as follows:
- fcAE16:
    - elayer1: 784 -> 256
    - ebatch1: 256 -> 256
    - elayer2: 256 -> 128
    - ebatch2: 128 -> 128
    - elayer3: 128 -> 16
    - dlayer1: 16 -> 128
    - dbatch1: 128 -> 128
    - dlayer2: 128 -> 256
    - dbatch2: 256 -> 256
    - dlayer3: 256 -> 784
- fcAE32:
    - elayer1: 784 -> 256
    - ebatch1: 256 -> 256
    - elayer2: 256 -> 128
    - ebatch2: 128 -> 128
    - elayer3: 128 -> 32
    - dlayer1: 32 -> 128
    - dbatch1: 128 -> 128
    - dlayer2: 128 -> 256
    - dbatch2: 256 -> 256
    - dlayer3: 256 -> 784
- fcAE128:
    - elayer1: 784 -> 256
    - ebatch1: 256 -> 256
    - elayer2: 256 -> 128
    - ebatch2: 128 -> 128
    - elayer3: 128 -> 128
    - dlayer1: 128 -> 128
    - dbatch1: 128 -> 128
    - dlayer2: 128 -> 256
    - dbatch2: 256 -> 256
    - dlayer3: 256 -> 784
- convAE16:
    - econv1: 1x28x28 -> 32x24x24 = 784 -> 18432
    - ebatch1: 32x24x24 -> 32x24x24 = 18432 -> 18432
    - econv2: 32x24x24 -> 8x20x20 = 18432 -> 3200
    - ebatch2: 8x20x20 -> 8x20x20 = 3200 -> 3200
    - econv3: 8x20x20 -> 16x1x1 = 3200 -> 16
    - dconv1: 16x1x1 -> 8x20x20 = 16 -> 3200
    - dbatch1: 8x20x20 -> 8x20x20 = 3200 -> 3200
    - dconv2: 8x20x20 -> 32x24x24 = 3200 -> 18432
    - dbatch2: 32x24x24 -> 32x24x24 = 18432 -> 18432
    - dconv3: 32x24x24 -> 1x28x28 = 18432 -> 784
- convAE32:
    - econv1: 1x28x28 -> 32x24x24 = 784 -> 18432
    - ebatch1: 32x24x24 -> 32x24x24 = 18432 -> 18432
    - econv2: 32x24x24 -> 8x20x20 = 18432 -> 3200
    - ebatch2: 8x20x20 -> 8x20x20 = 3200 -> 3200
    - econv3: 8x20x20 -> 32x1x1 = 3200 -> 32
    - dconv1: 32x1x1 -> 8x20x20 = 32 -> 3200
    - dbatch1: 8x20x20 -> 8x20x20 = 3200 -> 3200
    - dconv2: 8x20x20 -> 32x24x24 = 3200 -> 18432
    - dbatch2: 32x24x24 -> 32x24x24 = 18432 -> 18432
    - dconv3: 32x24x24 -> 1x28x28 = 18432 -> 784
- convAE128:
    - econv1: 1x28x28 -> 32x24x24 = 784 -> 18432
    - ebatch1: 32x24x24 -> 32x24x24 = 18432 -> 18432
    - econv2: 32x24x24 -> 8x20x20 = 18432 -> 3200
    - ebatch2: 8x20x20 -> 8x20x20 = 3200 -> 3200
    - econv3: 8x20x20 -> 128x1x1 = 3200 -> 128
    - dconv1: 128x1x1 -> 8x20x20 = 128 -> 3200
    - dbatch1: 8x20x20 -> 8x20x20 = 3200 -> 3200 
    - dconv2: 8x20x20 -> 32x24x24 = 3200 -> 18432
    - dbatch2: 32x24x24 -> 32x24x24 = 18432 -> 18432
    - dconv3: 32x24x24 -> 1x28x28 = 18432 -> 784

#### For the convolution autoencoder, the ranges are represented as the dimensions of the layer and number of feature maps. These values can be multiplied to get the total number of dimensions in the input and output for the layer. The layers prefixed with 'e' are the encoder layers and the layers prefixed with 'd' are the decoder layers.
 
#### By looking at the weights alone, It is possible to tell whether the encoder is a submersion or not. However this requires certain things to be true to be possible. Firstly, the activation function we are using should be a linear activation function, with the first derivative being a constant. In this case we are making use of the ELU activation function which is equal to x at x>0 whose first derivative is a constant. so assuming that the situation where the value passed into the activation function is always greater than zero the differential is a constant. This means that the jacobian matrix of a single layer in the encoder is going to be a constant matrix. And if this constant matrix has a full rank can be purely decided using the weights of the layer. If the weights of the layer are such that the jacobian matrix has full rank then that layer of the encoder is a submersion provided that the input dimension of the layer is greater than the output dimension. And in this way if an encoder consists of only layer which have these kind of activation functions and weights that give a full rank matrix with the input dimension being greater than or equal to the output dimension then the encoder is a submersion because that encoder is a composition of these submersion layers and theoretically the composition of submersion layers is a submersion and the encoder is a submersion. This means that the by only looking at the weights on the encoder irrespective of the input data points, we can determine if the encoder is a submersion or not. But please note that this is just a theoretical explanation involving a lot of assumptions and in practice, usually we would need to look at the data to determine if the encoder is a submersion or not at each particular point and we cannot determine if the encoder is a submersion or not just by looking at the weights.

#### Similar to the above explanation about the encoder, the decoder can also be an immersion if it is a composition of immersion functions. The decoder can be an immersion irrespective of the data points if it is composed of layers having a constant full rank jacobian and the input layer is smaller than or equal to the dimensions of the output layer. This is because theoretically the composition of immersion functions is an immersion. So by looking at the weights of the decoder, we can determine if the decoder is an immersion or not. However, this is a theoretical explanation and in practice, in most cases that don't have all these assumptions we would need to look at the data to determine if the decoder is an immersion or not at each particular point and we cannot determine if the decoder is an immersion or not just by looking at the weights.





### Part 6
Write a function to compute the Jacobian matrix of the decoder of a given model. It should
take as input the data point (image) at which to compute the Jacobian matrix. For each
model, compute the Jacobian of the decoder at a particular encoded data point. Compute
the SVD of the Jacobian and report the min and max singular values. Does it seem that the
Jacobian is full rank? What does this say about whether the decoder is an immersion at this
point?

In [ ]:
# Function to compute the Jacobian matrix of the decoder at a given point z
def compute_jacobian(model, z):
    # Store the original shape of z
    z_shape = z.shape  # Could be [latent_dim] or [channels, height, width]
    z_numel = z.numel()  # Total number of elements in z

    # Flatten z to a 1D tensor and ensure it requires gradient
    z_flat = z.clone().detach().view(-1).requires_grad_(True)  # [z_numel]

    # Define the decoder function for a single data point
    def decoder_func(z_input_flat):
        # Reshape z_input_flat back to z's original shape
        z_input = z_input_flat.view(z_shape)  # [latent_dim] or [channels, H, W]
        # Add batch dimension
        if isinstance(model, FullyConnectedAutoencoder):
            z_input = z_input.unsqueeze(0)  # [1, latent_dim]
        else:
            z_input = z_input.unsqueeze(0)  # [1, channels, H, W]
        y = model.decoder(z_input)  # Output shape: [1, C_out, H_out, W_out]
        y = y.view(-1)  # Flatten output to 1D tensor [output_dim]
        return y

    # Compute the Jacobian matrix
    jacobian = torch.autograd.functional.jacobian(decoder_func, z_flat, create_graph=False)

    # Convert Jacobian to a 2D tensor: [output_dim, z_numel]
    jacobian = jacobian.view(-1, z_numel)

    return jacobian

# Now, for each model, compute the Jacobian and its SVD
models_list = ['fcAE16', 'fcAE32', 'fcAE128', 'convAE16', 'convAE32', 'convAE128']

# Select a test image and encode it
test_image = test_data[0][0].unsqueeze(0).to(device)  # [1, 1, 28, 28]

for model_name in models_list:
    model = models[model_name]
    model.eval()

    # Get the encoded representation z
    with torch.no_grad():
        if isinstance(model, FullyConnectedAutoencoder):
            x = test_image.view(-1, 28*28)  # Flatten image: [1, 784]
            z = model.encoder(x).squeeze(0)  # [latent_dim]
        else:
            z = model.encoder(test_image).squeeze(0)  # [channels, H, W]

    # Compute Jacobian
    jacobian = compute_jacobian(model, z)

    # Convert Jacobian to NumPy array
    jacobian_np = jacobian.cpu().detach().numpy()

    # Compute SVD
    U, s, Vh = np.linalg.svd(jacobian_np, full_matrices=False)

    # Report results
    print(f"Model: {model_name}")
    print(f"Jacobian shape: {jacobian_np.shape}")
    print(f"Min singular value: {s.min()}")
    print(f"Max singular value: {s.max()}")
    print(f"Rank of Jacobian: {np.linalg.matrix_rank(jacobian_np)}")
    print(f"Is Jacobian full rank? {'Yes' if np.linalg.matrix_rank(jacobian_np) == min(jacobian_np.shape) else 'No'}")
    print(f"Decoder is an immersion at this point? {'Yes' if np.linalg.matrix_rank(jacobian_np) == z.numel() else 'No'}")
    print("-" * 50)

### Part 7
Pick an image as a base point. For each model, plot (as a grid of images) the tangent vectors
corresponding to the latent coordinate axes. Also, include a plot of the tangent vectors
corresponding to the principal components up to 128 dimensions. Describe what you are
seeing. Do the latent dimensions make sense in terms of changes to the base image?

In [ ]:
# Function to compute the Jacobian matrix of the decoder at a given point z
def compute_jacobian(model, z):
    # Store the original shape of z
    z_shape = z.shape  # Could be [latent_dim] or [channels, height, width]
    z_numel = z.numel()  # Total number of elements in z
    
    # Flatten z to a 1D tensor and ensure it requires gradient
    z_flat = z.clone().detach().view(-1).requires_grad_(True)  # [z_numel]
    
    # Define the decoder function for a single data point
    def decoder_func(z_input_flat):
        # Reshape z_input_flat back to z's original shape
        z_input = z_input_flat.view(z_shape)  # [latent_dim] or [channels, H, W]
        # Add batch dimension
        if isinstance(model, FullyConnectedAutoencoder):
            z_input = z_input.unsqueeze(0)  # [1, latent_dim]
        else:
            z_input = z_input.unsqueeze(0)  # [1, channels, H, W]
        y = model.decoder(z_input)  # Output shape: [1, C_out, H_out, W_out]
        y = y.view(-1)  # Flatten output to 1D tensor [output_dim]
        return y
    
    # Compute the Jacobian matrix
    jacobian = torch.autograd.functional.jacobian(decoder_func, z_flat)
    
    # Reshape Jacobian to 2D tensor: [output_dim, z_numel]
    jacobian = jacobian.view(-1, z_numel)
    
    return jacobian

# Function to plot the tangent vectors (Jacobian columns)
def plot_tangent_vectors(jacobian, model_name):
    # jacobian: [output_dim, z_numel]
    z_numel = jacobian.shape[1]
    images = []
    for i in range(z_numel):
        tangent_vector = jacobian[:, i]  # [output_dim]
        # Normalize for visualization
        tangent_vector = tangent_vector - tangent_vector.min()
        tangent_vector = tangent_vector / tangent_vector.max()
        image = tangent_vector.reshape(28, 28).detach().cpu().numpy()
        images.append(image)
    # Plot images in a grid
    cols = int(np.ceil(np.sqrt(z_numel)))
    rows = int(np.ceil(z_numel / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    fig.suptitle(f"Tangent Vectors (Jacobian Columns) - {model_name}", fontsize=16)
    for idx, ax in enumerate(axes.flatten()):
        if idx < z_numel:
            ax.imshow(images[idx], cmap='gray')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

# Select a base image
base_image = test_data[0][0].unsqueeze(0).to(device)  # [1, 1, 28, 28]

# List of models
models_list = ['fcAE16', 'fcAE32', 'fcAE128', 'convAE16', 'convAE32', 'convAE128']

for model_name in models_list:
    model = models[model_name]
    model.eval()
    
    # Get the encoded representation z
    with torch.no_grad():
        if isinstance(model, FullyConnectedAutoencoder):
            x = base_image.view(-1, 28*28)  # Flatten image: [1, 784]
            z = model.encoder(x).squeeze(0)  # [latent_dim]
        else:
            z = model.encoder(base_image).squeeze(0)  # [channels, H, W]
    
    # Compute Jacobian
    jacobian = compute_jacobian(model, z)
    
    # Plot tangent vectors
    plot_tangent_vectors(jacobian, model_name)

# For PCA
# Fit PCA on the training images flattened
train_images_pca = train_images_flat.numpy()  # [N, D]
pca_n_components = 128
pca = PCA(n_components=pca_n_components)
pca.fit(train_images_pca)

components = pca.components_  # [n_components, D]

# Function to plot PCA components
def plot_pca_components(components, k):
    images = []
    for i in range(k):
        component = components[i]  # [D]
        # Normalize for visualization
        component = component - component.min()
        component = component / component.max()
        image = component.reshape(28, 28)
        images.append(image)
    # Plot images in a grid
    cols = int(np.ceil(np.sqrt(k)))
    rows = int(np.ceil(k / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    fig.suptitle(f"Principal Components (Tangent Vectors) - Top {k}", fontsize=16)
    for idx, ax in enumerate(axes.flatten()):
        if idx < k:
            ax.imshow(images[idx], cmap='gray')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

for k in [16, 32, 128]:
    plot_pca_components(components, k)

#### The latent dimensions represent the different possible changes in the base image, such as the different features of the base image, for example, in the case of a shoe, the latent dimensions represent the different regions of shoe that can exhibit changes, such as having a different colour or something different than the base image. So, yes in this scenario with our given base image, the latent dimensions do make sense. In the case of PCA the different principal components represent the different directions from the mean image in which the data varies more and each of the principal components represent the different features of all the images from the dataset. They also do make sense because they mean that was our data point translates to the latent space where all the nearby data points are similar to the base image, implied by our tangent vectors which are all in different directions.

In [ ]:
# Select the base image
base_image = test_data[0][0].unsqueeze(0).to(device)  # [1, 1, 28, 28]
base_image_np = base_image.cpu().numpy().squeeze(0).squeeze(0)  # [28, 28]
x0 = base_image_np.flatten()  # [784]

# Prepare the training data for PCA
train_images_flat_np = train_images_flat.numpy()  # [N, 784]

# Fit PCA on the training data
pca_n_components = 16
pca = PCA(n_components=pca_n_components)
pca.fit(train_images_flat_np)

components = pca.components_  # [n_components, 784]

# Project the base image onto the PCA components to get the coefficients
alpha = pca.transform(x0.reshape(1, -1))  # [1, n_components]

# epsilon value for perturbation
epsilon = 10

# Number of principal components to visualize
k = pca_n_components  # Can be set to 16, 32, 128

# Visualize the effect of moving along each principal component at the base image
def plot_pca_tangent_vectors_at_base(x0, components, epsilon, k):
    images = []
    for i in range(k):
        # Tangent vector (principal component)
        pc_i = components[i]  # [784]
        
        # Perturbed image: x = x0 + ε * pc_i
        x = x0 + epsilon * pc_i
        
        # normalize so that the pixel values are in [0, 1]
        x = (x - x.min()) / (x.max() - x.min())
        
        # Reshape back to [28, 28]
        image = x.reshape(28, 28)
        
        images.append(image)
    
    # Plot images in a grid
    cols = int(np.ceil(np.sqrt(k)))
    rows = int(np.ceil(k / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    fig.suptitle(f"PCA Tangent Vectors at Base Image - Top {k}", fontsize=16)
    for idx, ax in enumerate(axes.flatten()):
        if idx < k:
            ax.imshow(images[idx], cmap='gray')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

# Plot the tangent vectors at the base image for the top k principal components
plot_pca_tangent_vectors_at_base(x0, components, epsilon, k)

In [ ]:
# Select the base image
base_image = test_data[0][0].unsqueeze(0).to(device)  # [1, 1, 28, 28]
base_image_np = base_image.cpu().numpy().squeeze(0).squeeze(0)  # [28, 28]
x0 = base_image_np.flatten()  # [784]

# Prepare the training data for PCA
train_images_flat_np = train_images_flat.numpy()  # [N, 784]

# Fit PCA on the training data
pca_n_components = 32
pca = PCA(n_components=pca_n_components)
pca.fit(train_images_flat_np)

components = pca.components_  # [n_components, 784]

# Project the base image onto the PCA components to get the coefficients
alpha = pca.transform(x0.reshape(1, -1))  # [1, n_components]

# epsilon value for perturbation
epsilon = 10

# Number of principal components to visualize
k = pca_n_components  # Can be set to 16, 32, 128

# Visualize the effect of moving along each principal component at the base image
def plot_pca_tangent_vectors_at_base(x0, components, epsilon, k):
    images = []
    for i in range(k):
        # Tangent vector (principal component)
        pc_i = components[i]  # [784]
        
        # Perturbed image: x = x0 + ε * pc_i
        x = x0 + epsilon * pc_i
        
        # normalize so that the pixel values are in [0, 1]
        x = (x - x.min()) / (x.max() - x.min())
        
        # Reshape back to [28, 28]
        image = x.reshape(28, 28)
        
        images.append(image)
    
    # Plot images in a grid
    cols = int(np.ceil(np.sqrt(k)))
    rows = int(np.ceil(k / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    fig.suptitle(f"PCA Tangent Vectors at Base Image - Top {k}", fontsize=16)
    for idx, ax in enumerate(axes.flatten()):
        if idx < k:
            ax.imshow(images[idx], cmap='gray')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

# Plot the tangent vectors at the base image for the top k principal components
plot_pca_tangent_vectors_at_base(x0, components, epsilon, k)

In [ ]:
# Select the base image
base_image = test_data[0][0].unsqueeze(0).to(device)  # [1, 1, 28, 28]
base_image_np = base_image.cpu().numpy().squeeze(0).squeeze(0)  # [28, 28]
x0 = base_image_np.flatten()  # [784]

# Prepare the training data for PCA
train_images_flat_np = train_images_flat.numpy()  # [N, 784]

# Fit PCA on the training data
pca_n_components = 128
pca = PCA(n_components=pca_n_components)
pca.fit(train_images_flat_np)

components = pca.components_  # [n_components, 784]

# Project the base image onto the PCA components to get the coefficients
alpha = pca.transform(x0.reshape(1, -1))  # [1, n_components]

# epsilon value for perturbation
epsilon = 10

# Number of principal components to visualize
k = pca_n_components  # Can be set to 16, 32, 128

# Visualize the effect of moving along each principal component at the base image
def plot_pca_tangent_vectors_at_base(x0, components, epsilon, k):
    images = []
    for i in range(k):
        # Tangent vector (principal component)
        pc_i = components[i]  # [784]
        
        # Perturbed image: x = x0 + ε * pc_i
        x = x0 + epsilon * pc_i
        
        # normalize so that the pixel values are in [0, 1]
        x = (x - x.min()) / (x.max() - x.min())
        
        # Reshape back to [28, 28]
        image = x.reshape(28, 28)
        
        images.append(image)
    
    # Plot images in a grid
    cols = int(np.ceil(np.sqrt(k)))
    rows = int(np.ceil(k / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    fig.suptitle(f"PCA Tangent Vectors at Base Image - Top {k}", fontsize=16)
    for idx, ax in enumerate(axes.flatten()):
        if idx < k:
            ax.imshow(images[idx], cmap='gray')
            ax.axis('off')
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

# Plot the tangent vectors at the base image for the top k principal components
plot_pca_tangent_vectors_at_base(x0, components, epsilon, k)

#### We also display the different tangent vectors for the different principal components at the base image, We do this by adding a petrubation to the base image in the directions of the principal components. This makes sense as well because the top principal components directions show another class appearing in the image and maybe that's the region of points the principal component is pointing to.

### Part 8
Use the same base image, and again repeat the following for all models and PCA. Check
how close translations and rotations are to being represented in the tangent space, as follows:
Generate tangent vectors induced by x and y translation and rotation. Compute the angles
from these tangent vectors to the model’s tangent space.

In [ ]:
import torchvision.transforms.functional as TF

# Function to compute the angle between two vectors
def compute_angle(v1, v2):
    v1 = v1.flatten()
    v2 = v2.flatten()
    cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    # Clamp cos_theta to [-1, 1] to avoid numerical errors
    cos_theta = np.clip(cos_theta, -1, 1)
    angle = np.arccos(cos_theta)
    angle_degrees = np.degrees(angle)
    return angle_degrees

# Function to project vector v onto the column space of matrix J
def project_onto_tangent_space(J, v):
    # J: [output_dim, latent_dim]
    # v: [output_dim]
    # Compute (J^T J)
    JTJ = np.dot(J.T, J)
    # Compute pseudo-inverse of (J^T J)
    JTJ_inv = np.linalg.pinv(JTJ)
    # Compute projection coefficients: c = (J^T J)^{-1} J^T v
    c = np.dot(JTJ_inv, np.dot(J.T, v))
    # Compute the projection: proj_v = J c
    proj_v = np.dot(J, c)
    return proj_v

# Select the base image
base_image, _ = test_data[0]
base_image = base_image.unsqueeze(0).to(device)  # [1, 1, 28, 28]

# Generate induced tangent vectors
def generate_induced_tangent_vectors(base_image):
    # Parameters for small transformations
    delta = 5  # small pixel shift for translation
    angle_degrees = 25  # small rotation angle in degrees
    angle_radians = np.deg2rad(angle_degrees)  # convert to radians
    
    # Base image in PIL format
    base_image_pil = TF.to_pil_image(base_image.cpu().squeeze(0))
    
    # Translation in x
    trans_x_image = TF.affine(base_image_pil, angle=0.0, translate=(delta, 0.0), scale=1.0, shear=0.0)
    trans_x_tensor = TF.to_tensor(trans_x_image).to(device)
    t_x = ((trans_x_tensor - base_image) / delta).cpu().numpy().flatten()
    
    # Translation in y
    trans_y_image = TF.affine(base_image_pil, angle=0.0, translate=(0.0, delta), scale=1.0, shear=0.0)
    trans_y_tensor = TF.to_tensor(trans_y_image).to(device)
    t_y = ((trans_y_tensor - base_image) / delta).cpu().numpy().flatten()
    
    # Rotation
    rot_image = TF.affine(base_image_pil, angle=angle_degrees, translate=(0.0, 0.0), scale=1.0, shear=0.0)
    rot_tensor = TF.to_tensor(rot_image).to(device)
    t_rot = ((rot_tensor - base_image) / angle_radians).cpu().numpy().flatten()
    
    return t_x, t_y, t_rot

t_x, t_y, t_rot = generate_induced_tangent_vectors(base_image)

# Prepare PCA tangent space
train_images_flat_np = train_images_flat.numpy()  # [N, 784]
pca_n_components = 128
pca = PCA(n_components=pca_n_components)
pca.fit(train_images_flat_np)
components = pca.components_.T  # [784, n_components]

# Function to compute angles for PCA
def compute_angles_pca(t_x, t_y, t_rot, components, k):
    # Components: [784, k]
    components_k = components[:, :k]  # [784, k]
    # Project induced tangent vectors onto PCA tangent space
    proj_t_x = project_onto_tangent_space(components_k, t_x)
    proj_t_y = project_onto_tangent_space(components_k, t_y)
    proj_t_rot = project_onto_tangent_space(components_k, t_rot)
    # Compute angles
    angle_t_x = compute_angle(t_x, proj_t_x)
    angle_t_y = compute_angle(t_y, proj_t_y)
    angle_t_rot = compute_angle(t_rot, proj_t_rot)
    return angle_t_x, angle_t_y, angle_t_rot

# Function to compute angles for autoencoder models
def compute_angles_model(model, t_x, t_y, t_rot):
    model.eval()
    # Get the encoded representation z
    with torch.no_grad():
        if isinstance(model, FullyConnectedAutoencoder):
            x = base_image.view(-1, 28*28)  # Flatten image: [1, 784]
            z = model.encoder(x).squeeze(0)  # [latent_dim]
        else:
            z = model.encoder(base_image).squeeze(0)  # [channels, H, W]

    # Compute Jacobian
    jacobian = compute_jacobian(model, z)  # [784, z_numel]
    jacobian_np = jacobian.cpu().detach().numpy()  # [784, z_numel]

    # Project induced tangent vectors onto model's tangent space
    proj_t_x = project_onto_tangent_space(jacobian_np, t_x)
    proj_t_y = project_onto_tangent_space(jacobian_np, t_y)
    proj_t_rot = project_onto_tangent_space(jacobian_np, t_rot)

    # Compute angles
    angle_t_x = compute_angle(t_x, proj_t_x)
    angle_t_y = compute_angle(t_y, proj_t_y)
    angle_t_rot = compute_angle(t_rot, proj_t_rot)

    return angle_t_x, angle_t_y, angle_t_rot

# Compute angles for PCA
for k in [16, 32, 128]:
    angle_t_x, angle_t_y, angle_t_rot = compute_angles_pca(t_x, t_y, t_rot, components, k)
    print(f"PCA with k={k}:")
    print(f"Angle between t_x and tangent space: {angle_t_x:.2f} degrees")
    print(f"Angle between t_y and tangent space: {angle_t_y:.2f} degrees")
    print(f"Angle between t_rot and tangent space: {angle_t_rot:.2f} degrees")
    print("-" * 50)

# Compute angles for each model
models_list = ['fcAE16', 'fcAE32', 'fcAE128', 'convAE16', 'convAE32', 'convAE128']

for model_name in models_list:
    model = models[model_name]
    angle_t_x, angle_t_y, angle_t_rot = compute_angles_model(model, t_x, t_y, t_rot)
    print(f"Model: {model_name}")
    print(f"Angle between t_x and tangent space: {angle_t_x:.2f} degrees")
    print(f"Angle between t_y and tangent space: {angle_t_y:.2f} degrees")
    print(f"Angle between t_rot and tangent space: {angle_t_rot:.2f} degrees")
    print("-" * 50)

#### You can clearly observe that the angles between the tangent vectors and the tangent space decrease as the sizes of the model increase or the number of dimensions in the PCA increase. this result seems as expected because the larger models should be invariant to translation and rotation since they are going to be capturing more information and generalizing better.

### Part 9
Now repeat the previous rotation and translation experiment, but using the model
convAE32 aug.pth, which was trained with data augmentation of randomly rotated and
translated versions of the images. Did data augmentation help the model learn rotations
and translations (according to the tangent space measurements)?

In [ ]:
# Load the augmented model
convAE32_aug = ConvolutionalAutoencoder(32)
convAE32_aug.load_state_dict(torch.load("hw3/convAE32_aug.pth", map_location=device))
convAE32_aug = convAE32_aug.to(device)
convAE32_aug.eval()

# Add the augmented model to the models dictionary
models['convAE32_aug'] = convAE32_aug

# Compute angles for the augmented model
model_name = 'convAE32_aug'
model = models[model_name]
angle_t_x, angle_t_y, angle_t_rot = compute_angles_model(model, t_x, t_y, t_rot)
print(f"Model: {model_name}")
print(f"Angle between t_x and tangent space: {angle_t_x:.2f} degrees")
print(f"Angle between t_y and tangent space: {angle_t_y:.2f} degrees")
print(f"Angle between t_rot and tangent space: {angle_t_rot:.2f} degrees")
print("-" * 50)

# For comparison, print the results for convAE32 without augmentation
model_name = 'convAE32'
model = models[model_name]
angle_t_x_no_aug, angle_t_y_no_aug, angle_t_rot_no_aug = compute_angles_model(model, t_x, t_y, t_rot)
print(f"Model: {model_name} (without augmentation)")
print(f"Angle between t_x and tangent space: {angle_t_x_no_aug:.2f} degrees")
print(f"Angle between t_y and tangent space: {angle_t_y_no_aug:.2f} degrees")
print(f"Angle between t_rot and tangent space: {angle_t_rot_no_aug:.2f} degrees")
print("-" * 50)

#### According to the tangent space measurements, The augmented model does not seem to be performing better than the similar sized model without augmentation. Therefore in this case, for this point we can conclude that the data augmentation did not help the model learn rotations and translations better because of a lack of conclusive evidence for that fact. However it could be different for other points in the latent space. 

Programming Assisted By Github Copilot and ChatGPT

# End Of Homework-3

In [21]:
import torch
import numpy as np
import torchvision.transforms.functional as TF
import math
from tqdm import tqdm  # For progress bar

# Function to compute the angle between two vectors
def compute_angle(v1, v2):
    v1 = v1.flatten()
    v2 = v2.flatten()
    cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    # Clamp cos_theta to [-1, 1] to avoid numerical errors
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    angle = np.arccos(cos_theta)
    angle_degrees = np.degrees(angle)
    return angle_degrees

# Function to project vector v onto the column space of matrix J
def project_onto_tangent_space(J, v):
    # J: [output_dim, latent_dim]
    # v: [output_dim]
    # Compute (J^T J)
    JTJ = np.dot(J.T, J)
    # Compute pseudo-inverse of (J^T J)
    JTJ_inv = np.linalg.pinv(JTJ)
    # Compute projection coefficients: c = (J^T J)^{-1} J^T v
    c = np.dot(JTJ_inv, np.dot(J.T, v))
    # Compute the projection: proj_v = J c
    proj_v = np.dot(J, c)
    return proj_v

# Generate induced tangent vectors with division by transformation amount
def generate_induced_tangent_vectors(image):
    # Parameters for small transformations
    delta = 1.0  # small pixel shift for translation
    angle_degrees = 5.0  # small rotation angle in degrees
    angle_radians = np.deg2rad(angle_degrees)  # convert to radians

    # Convert image tensor to PIL Image
    image_pil = TF.to_pil_image(image.cpu().squeeze(0))

    # Translation in x
    trans_x_image = TF.affine(image_pil, angle=0.0, translate=(delta, 0.0), scale=1.0, shear=0.0)
    trans_x_tensor = TF.to_tensor(trans_x_image).to(device)
    t_x = ((trans_x_tensor - image) / delta).cpu().numpy().flatten()

    # Translation in y
    trans_y_image = TF.affine(image_pil, angle=0.0, translate=(0.0, delta), scale=1.0, shear=0.0)
    trans_y_tensor = TF.to_tensor(trans_y_image).to(device)
    t_y = ((trans_y_tensor - image) / delta).cpu().numpy().flatten()

    # Rotation
    rot_image = TF.affine(image_pil, angle=angle_degrees, translate=(0.0, 0.0), scale=1.0, shear=0.0)
    rot_tensor = TF.to_tensor(rot_image).to(device)
    t_rot = ((rot_tensor - image) / angle_radians).cpu().numpy().flatten()

    return t_x, t_y, t_rot

# Function to compute angles for a single model and image
def compute_angles_model(model, image, t_x, t_y, t_rot):
    model.eval()
    # Get the encoded representation z
    with torch.no_grad():
        if isinstance(model, FullyConnectedAutoencoder):
            x = image.view(-1, 28*28)  # Flatten image: [1, 784]
            z = model.encoder(x).squeeze(0)  # [latent_dim]
        else:
            z = model.encoder(image).squeeze(0)  # [channels, H, W]

    # Compute Jacobian
    jacobian = compute_jacobian(model, z)  # [784, z_numel]
    jacobian_np = jacobian.cpu().detach().numpy()  # [784, z_numel]

    # Project induced tangent vectors onto model's tangent space
    proj_t_x = project_onto_tangent_space(jacobian_np, t_x)
    proj_t_y = project_onto_tangent_space(jacobian_np, t_y)
    proj_t_rot = project_onto_tangent_space(jacobian_np, t_rot)

    # Compute angles
    angle_t_x = compute_angle(t_x, proj_t_x)
    angle_t_y = compute_angle(t_y, proj_t_y)
    angle_t_rot = compute_angle(t_rot, proj_t_rot)

    return angle_t_x, angle_t_y, angle_t_rot

# Number of images to process (adjust as needed)
num_images = 100  # or len(test_data) for all images

# Prepare models list
models_list = ['fcAE16', 'fcAE32', 'fcAE128', 'convAE16', 'convAE32', 'convAE128']

# Add augmented model if available
if 'convAE32_aug' in models:
    models_list.append('convAE32_aug')

# Initialize dictionaries to store angles
angles_dict = {}
for model_name in models_list:
    angles_dict[model_name] = {'t_x': [], 't_y': [], 't_rot': []}

# Loop over images and models
for idx in tqdm(range(num_images), desc="Processing images"):
    image, _ = test_data[idx]
    image = image.unsqueeze(0).to(device)  # [1, 1, 28, 28]

    # Generate induced tangent vectors
    t_x, t_y, t_rot = generate_induced_tangent_vectors(image)

    # For each model, compute angles
    for model_name in models_list:
        model = models[model_name]
        try:
            angle_t_x, angle_t_y, angle_t_rot = compute_angles_model(model, image, t_x, t_y, t_rot)
        except RuntimeError as e:
            print(f"Error processing image {idx} with model {model_name}: {e}")
            continue  # Skip this image for this model

        angles_dict[model_name]['t_x'].append(angle_t_x)
        angles_dict[model_name]['t_y'].append(angle_t_y)
        angles_dict[model_name]['t_rot'].append(angle_t_rot)

# Compute average angles for each model
print("\nAverage Angles over all images:")
for model_name in models_list:
    avg_angle_t_x = np.mean(angles_dict[model_name]['t_x'])
    avg_angle_t_y = np.mean(angles_dict[model_name]['t_y'])
    avg_angle_t_rot = np.mean(angles_dict[model_name]['t_rot'])
    print(f"Model: {model_name}")
    print(f"Average angle between t_x and tangent space: {avg_angle_t_x:.2f} degrees")
    print(f"Average angle between t_y and tangent space: {avg_angle_t_y:.2f} degrees")
    print(f"Average angle between t_rot and tangent space: {avg_angle_t_rot:.2f} degrees")
    print("-" * 50)

Processing images: 100%|██████████| 100/100 [01:52<00:00,  1.13s/it]


Average Angles over all images:
Model: fcAE16
Average angle between t_x and tangent space: 62.96 degrees
Average angle between t_y and tangent space: 59.68 degrees
Average angle between t_rot and tangent space: 69.04 degrees
--------------------------------------------------
Model: fcAE32
Average angle between t_x and tangent space: 56.82 degrees
Average angle between t_y and tangent space: 53.00 degrees
Average angle between t_rot and tangent space: 63.75 degrees
--------------------------------------------------
Model: fcAE128
Average angle between t_x and tangent space: 43.37 degrees
Average angle between t_y and tangent space: 40.65 degrees
Average angle between t_rot and tangent space: 48.90 degrees
--------------------------------------------------
Model: convAE16
Average angle between t_x and tangent space: 56.24 degrees
Average angle between t_y and tangent space: 50.90 degrees
Average angle between t_rot and tangent space: 63.29 degrees
---------------------------------------